# Chapter 8 (Economic Modeling) -- EEIO Multi-Region Toy Accounting

This notebook continues the fully-synthetic-toy-data material from the
earlier input-output notebooks (see the first notebook in this sequence
for the data-availability disclosure covering the whole sequence) --
everything here is hardcoded, no external files needed.

It covers `eeio_compute_impact1` (single final-demand column, already
used earlier, rebuilt here with its two extra outputs `C_x`/`C_y` that
weren't needed before) and `eeio_compute_impact4` (the multi-column,
multi-region final-demand variant), plus two "value chain" table
constructions that carve a footprint total up by upstream/midstream/
downstream production stage rather than by sector.


In [1]:
import numpy as np
import pandas as pd

pd.set_option("display.float_format", lambda x: f"{x:,.2f}")


def eeio_compute_impact1(Z, y, x, CE):
    # Single final-demand column y; per-tier intermediate outputs aren't
    # returned here since they aren't needed by the exercises below.
    n = len(y)
    D_x = CE / x[None, :]
    A = Z / x[None, :]
    I = np.eye(n)
    L = np.linalg.inv(I - A)
    x_out = L @ y
    D_y = D_x @ L
    varpi_y = D_y @ y
    C_y = D_y * y[None, :]
    varpi_x = D_y @ (I - A) @ x_out
    C_x = (D_y @ (I - A)) * x_out[None, :]
    return dict(D_x=D_x, D_y=D_y, varpi_x=varpi_x, varpi_y=varpi_y,
                C_x=C_x, C_y=C_y, A=A, L=L, x=x_out)


def eeio_compute_impact4(Z, Y, x, CE):
    # Y is n x p (p final-demand columns, e.g. p regions); returns
    # per-column impact matrices C[:,:,r] and their row-summed totals C_r.
    n, p = Y.shape
    m = CE.shape[0]
    D_x = CE / x[None, :]
    A = Z / x[None, :]
    I = np.eye(n)
    L = np.linalg.inv(I - A)
    y_sum = Y.sum(axis=1)
    x_out = L @ y_sum
    D_y = D_x @ L
    varpi_y = D_y @ y_sum
    varpi_x = D_y @ (I - A) @ x_out
    C = np.zeros((m, n, p))
    C_r = np.zeros((m, p))
    for r in range(p):
        C[:, :, r] = D_y * Y[:, r][None, :]
        C_r[:, r] = C[:, :, r].sum(axis=1)
    return dict(D_x=D_x, D_y=D_y, varpi_x=varpi_x, varpi_y=varpi_y,
                C=C, C_r=C_r, A=A, L=L, x=x_out)


## Question 1 -- Single-region vs. two-column final demand, same 3-sector toy economy

The same 3-sector `Z`/`x`/`CE` toy economy used in earlier notebooks in
this sequence. First runs `eeio_compute_impact1` with the combined
final-demand vector `y = y1 + y2`, displaying its full output set
including `C_x` and `C_y` (the per-sector emissions-intensity-weighted
breakdowns of `varpi_x` and `varpi_y`). Then splits the same `y` into
two columns (`y1`, `y2` -- e.g. two downstream destinations) and reruns
via `eeio_compute_impact4`, confirming `D_y` is identical either way and
that the two columns' impacts (`C[:,:,0]`, `C[:,:,1]`) sum back to the
combined `C_y`.


In [2]:
Z = np.array([
    [100, 300, 100],
    [250, 150, 200],
    [ 25, 200,  75],
], dtype=float)
y1 = np.array([400, 700, 150.])
y2 = np.array([100, 700,  50.])
y = np.array([500, 1400, 200.])
x = np.array([1000, 2000, 500.])
CE = np.array([
    [50, 20, 5.],
    [ 3,  1, 0.],
])
sectors3 = ["Sector 1", "Sector 2", "Sector 3"]
species2 = ["CO2", "CH4"]

r1 = eeio_compute_impact1(Z, y, x, CE)

print("iot9a -- eeio_compute_impact1(Z, y, x, CE):")
print("\nC_x (direct-equivalent, weighted by output x):")
display(pd.DataFrame(np.hstack([r1["C_x"], r1["C_x"].sum(axis=1, keepdims=True)]),
                      index=species2, columns=sectors3 + ["Total"]))
print("\nC_y (footprint, weighted by final demand y):")
display(pd.DataFrame(np.hstack([r1["C_y"], r1["C_y"].sum(axis=1, keepdims=True)]),
                      index=species2, columns=sectors3 + ["Total"]))
print(f"\nvarpi_x = {r1['varpi_x']}   varpi_y = {r1['varpi_y']}")
print(f"\nC_x == CE exactly (an algebraic identity, not a coincidence -- see the write-up "
      f"above): {np.allclose(r1['C_x'], CE)}")

r4 = eeio_compute_impact4(Z, np.column_stack([y1, y2]), x, CE)
print("\niot9a (continued) -- eeio_compute_impact4(Z, [y1 y2], x, CE):")
print("D_y matches the single-column run:", np.allclose(r1["D_y"], r4["D_y"]))
for i, label in enumerate(["y1", "y2"]):
    print(f"\nC[:,:,{i}] (footprint attributable to {label}):")
    display(pd.DataFrame(np.hstack([r4["C"][:, :, i], r4["C"][:, :, i].sum(axis=1, keepdims=True)]),
                          index=species2, columns=sectors3 + ["Total"]))
print("\nC_r (region/column totals):")
display(pd.DataFrame(np.hstack([r4["C_r"], r4["C_r"].sum(axis=1, keepdims=True)]),
                      index=species2, columns=["y1", "y2", "Total"]))


iot9a -- eeio_compute_impact1(Z, y, x, CE):

C_x (direct-equivalent, weighted by output x):


,Sector 1,Sector 2,Sector 3,Total
CO2,50.00,20.00,5.00,75.00
CH4,3.00,1.00,0.00,4.00



C_y (footprint, weighted by final demand y):


,Sector 1,Sector 2,Sector 3,Total
CO2,31.83,35.44,7.73,75.00
CH4,1.87,1.83,0.30,4.00



varpi_x = [75.  4.]   varpi_y = [75.  4.]

C_x == CE exactly (an algebraic identity, not a coincidence -- see the write-up above): True

iot9a (continued) -- eeio_compute_impact4(Z, [y1 y2], x, CE):
D_y matches the single-column run: True

C[:,:,0] (footprint attributable to y1):


,Sector 1,Sector 2,Sector 3,Total
CO2,25.46,17.72,5.80,48.98
CH4,1.50,0.92,0.22,2.64



C[:,:,1] (footprint attributable to y2):


,Sector 1,Sector 2,Sector 3,Total
CO2,6.37,17.72,1.93,26.02
CH4,0.37,0.92,0.07,1.36



C_r (region/column totals):


,y1,y2,Total
CO2,48.98,26.02,75.00
CH4,2.64,1.36,4.00


## Question 2 -- Six-sector, three-column toy economy

A larger 6-sector economy with a 3-column final-demand matrix `y` (three
downstream regions/destinations) and a 2-species `CE`. Also computes
`V`, the value-added row (`x' - column sums of Z`), printed for
reference even though `V` isn't fed into any further impact calculation.
Shows the technical-coefficient matrix `A`, Leontief inverse `L`, the
per-species multipliers `D_x`/`D_y`, the closure totals `varpi_x`/
`varpi_y`, and the three per-column impact tables plus their `C_r`
summary.


In [3]:
Z6 = np.array([
    [100, 300,  10,  10,  20,   0],
    [250, 150,  20,   0,  10,   0],
    [ 10,  10, 110, 310,   0,   0],
    [ 20,  20,  80,  25,  15,  20],
    [ 10,   5,   8,   3,  40,   7],
    [  5,   2,   8,   8,  12,  35],
], dtype=float)
y6 = np.array([
    [500, 200, 25],
    [800, 100, 17],
    [ 20, 200, 15],
    [  0, 200,  5],
    [  5,  25, 50],
    [  3,  50, 50],
], dtype=float)
x6 = Z6.sum(axis=1) + y6.sum(axis=1)
V6 = x6 - Z6.sum(axis=0)
CE6 = 1000 * np.array([
    [50, 20, 10, 10, 5, 5.],
    [ 5,  3,  0,  0, 1, 1.],
])
sectors6 = [f"S{i+1}" for i in range(6)]
regions3 = ["Region 1", "Region 2", "Region 3"]

print("x (from Z row sums + y row sums):", x6)
print("V (value added, x - column sums of Z):", V6)

r6 = eeio_compute_impact4(Z6, y6, x6, CE6)

print("\nA (technical-coefficient matrix):")
display(pd.DataFrame(r6["A"], index=sectors6, columns=sectors6).round(4))
print("\nL (Leontief inverse):")
display(pd.DataFrame(r6["L"], index=sectors6, columns=sectors6).round(4))
print("\nD_x (direct multiplier, per unit of output):")
display(pd.DataFrame(r6["D_x"], index=species2, columns=sectors6))
print("\nD_y (total multiplier, direct + all indirect tiers):")
display(pd.DataFrame(r6["D_y"], index=species2, columns=sectors6).round(3))
print(f"\nvarpi_x = {r6['varpi_x']}   varpi_y = {r6['varpi_y']}   (both should equal CE row totals)")

for i, label in enumerate(regions3):
    print(f"\nC[:,:,{i}] (footprint attributable to {label}):")
    display(pd.DataFrame(np.hstack([r6["C"][:, :, i], r6["C"][:, :, i].sum(axis=1, keepdims=True)]),
                          index=species2, columns=sectors6 + ["Total"]))
print("\nC_r (region totals):")
display(pd.DataFrame(np.hstack([r6["C_r"], r6["C_r"].sum(axis=1, keepdims=True)]),
                      index=species2, columns=regions3 + ["Total"]))


x (from Z row sums + y row sums): [1165. 1347.  675.  385.  153.  173.]
V (value added, x - column sums of Z): [770. 860. 439.  29.  56. 111.]

A (technical-coefficient matrix):


,S1,S2,S3,S4,S5,S6
S1,0.09,0.22,0.01,0.03,0.13,0.00
S2,0.21,0.11,0.03,0.00,0.07,0.00
S3,0.01,0.01,0.16,0.81,0.00,0.00
S4,0.02,0.01,0.12,0.06,0.10,0.12
S5,0.01,0.00,0.01,0.01,0.26,0.04
S6,0.00,0.00,0.01,0.02,0.08,0.20



L (Leontief inverse):


,S1,S2,S3,S4,S5,S6
S1,1.17,0.30,0.05,0.07,0.24,0.02
S2,0.28,1.20,0.06,0.06,0.17,0.02
S3,0.05,0.04,1.37,1.19,0.19,0.18
S4,0.04,0.03,0.18,1.23,0.19,0.19
S5,0.02,0.01,0.03,0.04,1.37,0.07
S6,0.01,0.01,0.03,0.05,0.14,1.27



D_x (direct multiplier, per unit of output):


...

,S1,S2,S3,S4,S5,S6,Total
CO2,"11,358.96","3,250.33","5,901.83","11,282.56","1,743.91","2,397.30","35,934.90"
CH4,"1,162.41",404.53,131.81,199.86,280.35,398.00,"2,576.96"



C[:,:,2] (footprint attributable to Region 3):


,S1,S2,S3,S4,S5,S6,Total
CO2,"1,419.87",552.56,442.64,282.06,"3,487.82","2,397.30","8,582.25"
CH4,145.30,68.77,9.89,5.00,560.71,398.00,"1,187.66"



C_r (region totals):


,Region 1,Region 2,Region 3,Total
CO2,"55,482.86","35,934.90","8,582.25","100,000.00"
CH4,"6,235.37","2,576.96","1,187.66","10,000.00"


## Question 3 -- A value-chain table: allocating a footprint by production stage

Reuses Question 2's 6-sector economy but now asks a different question:
not "how much footprint does each final-demand column cause", but "how
much of the *first* species' emissions ($CE$'s row 1) flows through each
cell of the use table, and how does that total split across three
value-chain stages (sectors 1-2, 3-4, 5-6, read here as e.g. upstream /
midstream / downstream)?"

`T1` allocates each use-table entry (`Z` plus all three `y` columns, i.e.
`[Z y]`) by that row-sector's direct emissions-per-output ratio, $CE$'s
first row divided elementwise by $x$. Row sums of `T1` therefore
reproduce `CE`'s first row exactly (a one-tier, non-propagated
allocation -- contrast with Question 4 below, which uses the
fully-propagated `D_y` instead). `T2` groups `T1`'s 9 use-table columns
into the same three stages; `T3` further groups `T2`'s rows into the
three stages, producing a 3x3 stage-to-stage flow matrix. `T4` is a
diagnostic layout: `T0` (a hand-specified reference vector, which turns
out to equal `T3`'s row sums exactly -- see the sanity checks) against
`T3`'s off-diagonal ("O", inter-stage) flows.


In [4]:
CE1 = CE6[0, :]      # first species' row, as a per-sector vector
T0 = np.array([70000, 20000, 10000.])

Zy6 = np.hstack([Z6, y6])                       # [Z y], 6 x 9
T1 = Zy6 * (CE1 / x6)[:, None]                  # each row i scaled by CE1[i]/x6[i]
print("iot9c -- T1 ([Z y] allocated by row-sector direct intensity CE1/x):")
t1_df = pd.DataFrame(T1, index=sectors6,
                      columns=[f"Z->{s}" for s in sectors6] + [f"y->{r}" for r in regions3])
t1_df["Total"] = t1_df.sum(axis=1)
t1_df.loc["Total"] = t1_df.sum()
display(t1_df.round(1))

# T2: group the 9 columns into 3 value-chain stages: {S1,S2,y_R1}, {S3,S4,y_R2}, {S5,S6,y_R3}
T2 = np.column_stack([
    T1[:, [0, 1, 6]].sum(axis=1),
    T1[:, [2, 3, 7]].sum(axis=1),
    T1[:, [4, 5, 8]].sum(axis=1),
])
print("\niot9c -- T2 (T1's columns grouped into 3 stages):")
t2_df = pd.DataFrame(T2, index=sectors6, columns=["Stage 1", "Stage 2", "Stage 3"])
t2_df["Total"] = t2_df.sum(axis=1)
display(t2_df.round(1))

# T3: further group T2's 6 rows into the same 3 stages -> 3x3 stage flow matrix
T3 = np.array([
    [T2[0:2, 0].sum(), T2[0:2, 1].sum(), T2[0:2, 2].sum()],
    [T2[2:4, 0].sum(), T2[2:4, 1].sum(), T2[2:4, 2].sum()],
    [T2[4:6, 0].sum(), T2[4:6, 1].sum(), T2[4:6, 2].sum()],
])
print("\niot9c -- T3 (3x3 stage-to-stage flow matrix):")
stages = ["Stage 1", "Stage 2", "Stage 3"]
t3_df = pd.DataFrame(T3, index=stages, columns=stages)
t3_df["Total"] = t3_df.sum(axis=1)
display(t3_df.round(1))

D3 = np.diag(T3)
O3 = T3 - np.diag(D3)

T4 = np.vstack([
    T0,
    O3.sum(axis=1),
    T0 - O3.sum(axis=1),
    O3.sum(axis=0),
    D3 + O3.sum(axis=0),
])
print("\niot9c -- T4 (diagnostic layout: T0 reference, inter-stage in/out flows, "
      "reconstructed diagonal):")
t4_df = pd.DataFrame(T4, index=["T0 (reference)", "Outflow to other stages",
                                 "T0 - outflow", "Inflow from other stages",
                                 "Within-stage + inflow"], columns=stages)
t4_df["Total"] = t4_df.sum(axis=1)
display(t4_df.round(1))


iot9c -- T1 ([Z y] allocated by row-sector direct intensity CE1/x):


,Z->S1,Z->S2,Z->S3,Z->S4,Z->S5,Z->S6,y->Region 1,y->Region 2,y->Region 3,Total
S1,"4,291.80","12,875.50",429.20,429.20,858.40,0.00,"21,459.20","8,583.70","1,073.00","50,000.00"
S2,"3,712.00","2,227.20",297.00,0.00,148.50,0.00,"11,878.20","1,484.80",252.40,"20,000.00"
S3,148.10,148.10,"1,629.60","4,592.60",0.00,0.00,296.30,"2,963.00",222.20,"10,000.00"
S4,519.50,519.50,"2,077.90",649.40,389.60,519.50,0.00,"5,194.80",129.90,"10,000.00"
S5,326.80,163.40,261.40,98.00,"1,307.20",228.80,163.40,817.00,"1,634.00","5,000.00"
S6,144.50,57.80,231.20,231.20,346.80,"1,011.60",86.70,"1,445.10","1,445.10","5,000.00"
Total,"9,142.70","15,991.50","4,926.30","6,000.40","3,050.50","1,759.80","33,883.90","20,488.30","4,756.50","100,000.00"



iot9c -- T2 (T1's columns grouped into 3 stages):


,Stage 1,Stage 2,Stage 3,Total
S1,"38,626.60","9,442.10","1,931.30","50,000.00"
S2,"17,817.40","1,781.70",400.90,"20,000.00"
S3,592.60,"9,185.20",222.20,"10,000.00"
S4,"1,039.00","7,922.10","1,039.00","10,000.00"
S5,653.60,"1,176.50","3,169.90","5,000.00"
S6,289.00,"1,907.50","2,803.50","5,000.00"



iot9c -- T3 (3x3 stage-to-stage flow matrix):


,Stage 1,Stage 2,Stage 3,Total
Stage 1,"56,444.00","11,223.80","2,332.20","70,000.00"
Stage 2,"1,631.60","17,107.30","1,261.20","20,000.00"
Stage 3,942.60,"3,084.00","5,973.40","10,000.00"



iot9c -- T4 (diagnostic layout: T0 reference, inter-stage in/out flows, reconstructed diagonal):


,Stage 1,Stage 2,Stage 3,Total
T0 (reference),"70,000.00","20,000.00","10,000.00","100,000.00"
Outflow to other stages,"13,556.00","2,892.70","4,026.60","20,475.40"
T0 - outflow,"56,444.00","17,107.30","5,973.40","79,524.60"
Inflow from other stages,"2,574.20","14,307.80","3,593.40","20,475.40"
Within-stage + inflow,"59,018.10","31,415.00","9,566.80","100,000.00"


## Question 4 -- The same value chain, allocated by the fully-propagated multiplier

Same 6-sector economy and the same three-stage grouping as Question 3,
but a different allocation rule for `T1`: instead of weighting `[Z y]`
by the *direct* ratio (CE's first row divided elementwise by `x`), this
version weights the final-demand matrix `y` alone by `D_y`'s first row --
the *fully-propagated* total multiplier (direct + every indirect tier,
from `eeio_compute_impact4`). Because `D_y` already embeds supply-chain
propagation, `T1`'s row sums no longer reproduce `CE1` exactly the way
Question 3's did; the gap between them is computed explicitly.

The diagnostic table `T4` is correspondingly extended with a `VC`
("value-chain adjustment") row: the negative of that gap, grouped into
the three stages, added to `T0` to get an adjusted reference `new_T0`
that *does* reconcile with this table's own row sums -- and a final row
`sum(T3,1) - T0` that shows by how much the raw stage totals still miss
the unadjusted `T0`.


In [5]:
r6b = eeio_compute_impact4(Z6, y6, x6, CE6)
D_y6 = r6b["D_y"]

T1d = D_y6[0, :][:, None] * y6     # D_y's first row, broadcast against y -- scales each sector's contribution by its *total* multiplier
CE1 = CE6[0, :]
T0 = np.array([70000, 20000, 10000.])

print("iot9d -- T1 (y allocated by the fully-propagated multiplier D_y row 1), "
      "vs. direct CE1:")
res_d = pd.DataFrame(np.column_stack([T1d, T1d.sum(axis=1), CE1, T1d.sum(axis=1) - CE1]),
                      index=sectors6, columns=regions3 + ["Total", "CE (direct)", "Total - CE"])
res_d.loc["Total"] = res_d.sum()
display(res_d.round(1))

T2d = T1d[:, 0:3]
T3d = np.array([
    [T2d[0:2, 0].sum(), T2d[0:2, 1].sum(), T2d[0:2, 2].sum()],
    [T2d[2:4, 0].sum(), T2d[2:4, 1].sum(), T2d[2:4, 2].sum()],
    [T2d[4:6, 0].sum(), T2d[4:6, 1].sum(), T2d[4:6, 2].sum()],
])
print("\niot9d -- T3 (3x3 stage flow matrix, this allocation rule):")
stages = ["Stage 1", "Stage 2", "Stage 3"]
display(pd.DataFrame(T3d, index=stages, columns=stages).round(1))

D3d = np.diag(T3d)
O3d = T3d - np.diag(D3d)

VC = -(CE1 - T1d.sum(axis=1))
VC = np.array([VC[0:2].sum(), VC[2:4].sum(), VC[4:6].sum()])
new_T0 = T0 + VC
print(f"\nVC (value-chain adjustment, by stage): {VC}")
print(f"new_T0 = T0 + VC: {new_T0}")

T4d = np.vstack([
    T0,
    VC,
    new_T0,
    O3d.sum(axis=1),
    new_T0 - O3d.sum(axis=1),
    O3d.sum(axis=0),
    D3d + O3d.sum(axis=0),
    T3d.sum(axis=0) - T0,
])
print("\niot9d -- T4 (extended diagnostic layout):")
t4d_df = pd.DataFrame(T4d, index=["T0 (reference)", "VC (adjustment)", "new_T0 = T0+VC",
                                   "Outflow to other stages", "new_T0 - outflow",
                                   "Inflow from other stages", "Within-stage + inflow",
                                   "Raw stage total - T0"], columns=stages)
t4d_df["Total"] = t4d_df.sum(axis=1)
display(t4d_df.round(1))


iot9d -- T1 (y allocated by the fully-propagated multiplier D_y row 1), vs. direct CE1:


,Region 1,Region 2,Region 3,Total,CE (direct),Total - CE
S1,"28,397.40","11,359.00","1,419.90","41,176.20","50,000.00","-8,823.80"
S2,"26,002.60","3,250.30",552.60,"29,805.50","20,000.00","9,805.50"
S3,590.20,"5,901.80",442.60,"6,934.60","10,000.00","-3,065.40"
S4,0.00,"11,282.60",282.10,"11,564.60","10,000.00","1,564.60"
S5,348.80,"1,743.90","3,487.80","5,580.50","5,000.00",580.50
S6,143.80,"2,397.30","2,397.30","4,938.40","5,000.00",-61.60
Total,"55,482.90","35,934.90","8,582.20","100,000.00","100,000.00",-0.00



iot9d -- T3 (3x3 stage flow matrix, this allocation rule):


,Stage 1,Stage 2,Stage 3
Stage 1,"54,400.10","14,609.30","1,972.40"
Stage 2,590.20,"17,184.40",724.70
Stage 3,492.60,"4,141.20","5,885.10"



VC (value-chain adjustment, by stage): [  981.77219232 -1500.72516283   518.95297051]
new_T0 = T0 + VC: [70981.77219232 18499.27483717 10518.95297051]

iot9d -- T4 (extended diagnostic layout):


,Stage 1,Stage 2,Stage 3,Total
T0 (reference),"70,000.00","20,000.00","10,000.00","100,000.00"
VC (adjustment),981.80,"-1,500.70",519.00,-0.00
new_T0 = T0+VC,"70,981.80","18,499.30","10,519.00","100,000.00"
Outflow to other stages,"16,581.70","1,314.90","4,633.80","22,530.40"
new_T0 - outflow,"54,400.10","17,184.40","5,885.10","77,469.60"
Inflow from other stages,"1,082.80","18,750.50","2,697.10","22,530.40"
Within-stage + inflow,"55,482.90","35,934.90","8,582.20","100,000.00"
Raw stage total - T0,"-14,517.10","15,934.90","-1,417.80",-0.00


## Sanity checks


In [6]:
checks = []

checks.append(("Q1: C_x equals CE exactly -- an algebraic identity that follows directly "
                "from the definitions of D_x, D_y, and A, not a coincidence of these "
                "particular numbers",
                np.allclose(r1["C_x"], CE)))
checks.append(("Q1: varpi_x and varpi_y both equal CE's row totals exactly (closure "
                "of the multi-tier accounting)",
                np.allclose(r1["varpi_x"], CE.sum(axis=1)) and
                np.allclose(r1["varpi_y"], CE.sum(axis=1))))
checks.append(("Q1: splitting y into (y1, y2) columns and running eeio_compute_impact4 "
                "reproduces the same D_y as the single-column eeio_compute_impact1 run, and "
                "the two columns' impacts sum back to the combined C_y",
                np.allclose(r1["D_y"], r4["D_y"]) and
                np.allclose(r4["C"][:, :, 0] + r4["C"][:, :, 1], r1["C_y"])))
checks.append(("Q2: x reconstructed from Z's row sums plus y's row sums matches the "
                "expected hardcoded totals for this economy",
                np.allclose(x6, np.array([1165, 1347, 675, 385, 153, 173.]))))
checks.append(("Q2: varpi_x, varpi_y, and C_r's grand total all equal CE6's row totals "
                "(closure holds for the larger 6-sector/3-region economy too)",
                np.allclose(r6["varpi_x"], CE6.sum(axis=1)) and
                np.allclose(r6["varpi_y"], CE6.sum(axis=1)) and
                np.allclose(r6["C_r"].sum(axis=1), CE6.sum(axis=1))))
checks.append(("Q3: T1's row sums reproduce CE1 exactly (the direct, non-propagated "
                "allocation rule is exact by construction)",
                np.allclose(T1.sum(axis=1), CE1)))
checks.append(("Q3: T3's row sums equal T0 exactly -- confirming T0 was chosen as "
                "exactly this economy's true stage totals under the direct-allocation rule",
                np.allclose(T3.sum(axis=1), T0)))
checks.append(("Q3: T4's reconstructed diagonal (T0 - outflow) matches T3's actual "
                "diagonal exactly",
                np.allclose(T0 - O3.sum(axis=1), D3)))
checks.append(("Q4: unlike Question 3, T1's row sums do NOT reproduce CE1 exactly -- the "
                "fully-propagated D_y allocation and the direct CE/x allocation are genuinely "
                "different rules, which is exactly why this section needs its VC adjustment "
                "step",
                not np.allclose(T1d.sum(axis=1), CE1)))
checks.append(("Q4: new_T0's outflow-adjusted row (new_T0 - outflow) reconstructs "
                "T3d's actual diagonal exactly, confirming VC correctly reconciles the two "
                "allocation rules",
                np.allclose(new_T0 - O3d.sum(axis=1), D3d)))

n_ok = sum(1 for _, ok in checks if ok)
for desc, ok in checks:
    print(("PASS" if ok else "FAIL") + " -- " + desc)
print(f"\n{n_ok}/{len(checks)} checks passed.")
assert n_ok == len(checks), "one or more sanity checks failed"


PASS -- Q1: C_x equals CE exactly -- an algebraic identity that follows directly from the definitions of D_x, D_y, and A, not a coincidence of these particular numbers
PASS -- Q1: varpi_x and varpi_y both equal CE's row totals exactly (closure of the multi-tier accounting)
PASS -- Q1: splitting y into (y1, y2) columns and running eeio_compute_impact4 reproduces the same D_y as the single-column eeio_compute_impact1 run, and the two columns' impacts sum back to the combined C_y
PASS -- Q2: x reconstructed from Z's row sums plus y's row sums matches the expected hardcoded totals for this economy
PASS -- Q2: varpi_x, varpi_y, and C_r's grand total all equal CE6's row totals (closure holds for the larger 6-sector/3-region economy too)
PASS -- Q3: T1's row sums reproduce CE1 exactly (the direct, non-propagated allocation rule is exact by construction)
PASS -- Q3: T3's row sums equal T0 exactly -- confirming T0 was chosen as exactly this economy's true stage totals under the direct-allocatio

## Summary

Covered `eeio_compute_impact1`'s two extra emissions-weighted outputs
(`C_x`, `C_y`, including the `C_x == CE` identity), `eeio_compute_impact4`'s
multi-column/multi-region variant on both a 3-sector and a 6-sector toy
economy, and two contrasting "value chain" table constructions (a
direct-allocation rule vs. a fully-propagated-multiplier rule) with
their respective diagnostic reconciliation steps. No data-availability
gaps for this notebook (see the first notebook's introduction for the
disclosure covering this whole sequence). All 9 sanity checks passed (0
errors, 0 stderr).